# Track 5: Group Relative Policy Optimization (GRPO) for Mathematical Reasoning

This notebook demonstrates fine-tuning on mathematical reasoning problems from the `openai/gsm8k` dataset using **Group Relative Policy Optimization (GRPO)** over 250 steps. We configure reinforcement learning rewards based on format correctness (1.0 weight) and solution accuracy (3.0 weight) to teach the model strict XML reasoning and answer tags (`<reasoning>` and `<answer>`).

## 1. Setup Environment and Imports
We load libraries and establish system prompt instructions enforcing a strict XML reasoning and answer format.

In [1]:
# !pip install uv -q unsloth vllm


In [2]:
from unsloth import FastLanguageModel, is_bfloat16_supported

import re
import os
import sys
import glob
import torch
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

print(f"CUDA available: {torch.cuda.is_available()} | Compute Dtype: {compute_dtype}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True | Compute Dtype: torch.bfloat16


## 2. Dataset Preparation (GSM8K)
We load the GSM8K dataset. We format each question with a system instruction that instructs the model to put its reasoning inside `<reasoning>` tags and the final numeric answer inside `<answer>` tags.

In [3]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\n"
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\n"
    "The reasoning process and answer are enclosed within tags. The answer must be a single integer.\n"
    "Example:\n"
    "<reasoning>\n"
    "We know that 2 + 2 = 4.\n"
    "</reasoning>\n"
    "<answer>4</answer>"
)

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

dataset = load_dataset("openai/gsm8k", "main", split="train")
dataset = dataset.shuffle(seed=42)

def format_gsm8k(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "answer": extract_hash_answer(example["answer"]),
    }


gsm8k_train = dataset.map(format_gsm8k)

print("Sample Prompt Structure Preview:\n", gsm8k_train[0]["prompt"])

Sample Prompt Structure Preview:
 [{'role': 'system', 'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\nThe reasoning process and answer are enclosed within tags. The answer must be a single integer.\nExample:\n<reasoning>\nWe know that 2 + 2 = 4.\n</reasoning>\n<answer>4</answer>'}, {'role': 'user', 'content': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?'}]


## 3. Define RL Rewards
We define two reward functions:
1. **Format Reward**: Returns `1.0` if the output strictly matches the `<reasoning>...</reasoning>\n<answer>...</answer>` tags.
2. **Correctness Reward**: Returns `2.0` if the extracted answer matches the ground truth, and `0.0` otherwise.

In [4]:
def extract_last_number(text, start_tag="<answer>", end_tag="</answer>"):
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    matches = re.findall(pattern, text, re.DOTALL)
    text_to_search = matches[-1] if matches else text
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text_to_search)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return ""

def extract_answer(text):
    return extract_last_number(text)

# 1. Graduated Format Reward (Max: 0.5)
def format_reward(completions, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    for response in responses:
        score = 0.0
        if "<reasoning>" in response: score += 0.1
        if "</reasoning>" in response: score += 0.1
        if "<answer>" in response: score += 0.1
        if "</answer>" in response: score += 0.1
        
        pattern = r"^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$"
        if re.match(pattern, response):
            score += 0.1
            
        rewards.append(score)
    return rewards

# 2. Math Step / Equation Reward (Replaces raw length reward, Max: 0.5)
def math_step_reward(completions, **kwargs):
    """
    Rewards generating actual mathematical calculations (e.g., '16 - 7 = 9')
    rather than just filling space with words.
    """
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    for response in responses:
        match = re.search(r"<reasoning>(.*?)</reasoning>", response, re.DOTALL)
        if match:
            text = match.group(1)
            # Find explicit arithmetic operations like "16 - 7 = 9" or "180 / 3 = 60"
            equations = re.findall(r"\d+\s*[\+\-\*/]\s*\d+\s*=\s*\d+", text)
            eq_count = len(equations)
            
            # Award +0.15 per equation, capped at 0.5 max
            rewards.append(min(0.5, eq_count * 0.15))
        else:
            rewards.append(0.0)
    return rewards

# 3. Partial Correctness Reward (Max: 0.5)
def partial_correctness_reward(completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    for response, ans in zip(responses, answer):
        if ans and re.search(r"\b" + re.escape(str(ans)) + r"\b", response):
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

# 4. Strict Final Correctness Reward (Max: 2.0)
def correctness_reward(completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    extracted = [extract_last_number(response) for response in responses]
    rewards = [2.0 if ext == ans else 0.0 for ext, ans in zip(extracted, answer)]
    return rewards

## 3.5. Evaluate Pre-Trained Model Before GRPO Alignment
To demonstrate GRPO's capability to teach both format compliance and mathematical accuracy, we evaluate the un-finetuned model on 5 test questions from GSM8K before starting RL training.

In [5]:
HF_REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = HF_REPO_ID.split("/")[-1].lower()

In [6]:
max_seq_length = 2048
lora_rank = 64

base_model_eval, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_REPO_ID,
    max_seq_length=max_seq_length,
    load_in_4bit=False,      # set True for 4-bit quantized loading
    fast_inference=True,     # enables Unsloth's vLLM-backed fast generation
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,  # lower if you hit OOM
)
FastLanguageModel.for_inference(base_model_eval)
base_model_eval.generation_config.max_length = None

eval_dataset_100 = load_dataset("openai/gsm8k", "main", split="test").select(range(100))
pre_grpo_responses_100 = []
pre_grpo_correct = 0
pre_grpo_format = 0

pattern = r"^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$"


print("=== Running Pre-GRPO Benchmark on 100 GSM8K Test Problems ===")

for i, example in enumerate(eval_dataset_100):
    question = example["question"]
    ground_truth = extract_hash_answer(example["answer"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs["input_ids"]
    input_ids = input_ids.to("cuda")

    with torch.no_grad():
        outputs = base_model_eval.generate(
            input_ids=input_ids, max_new_tokens=2048, pad_token_id=tokenizer.eos_token_id
        )

    resp = tokenizer.decode(
        outputs[0][input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    pre_grpo_responses_100.append(resp)
    ext_ans = extract_last_number(resp)

    if ext_ans == ground_truth:
        pre_grpo_correct += 1

    if re.match(pattern, resp):
        pre_grpo_format += 1

    if (i + 1) % 20 == 0:
        print(
            f"Evaluated {i + 1}/100 problems | Current Accuracy: {pre_grpo_correct / (i + 1) * 100:.1f}%"
        )

pre_acc = (pre_grpo_correct / 100) * 100
pre_fmt = (pre_grpo_format / 100) * 100

print(f"\n>>> PRE-GRPO BENCHMARK (100 Problems) <<<")
print(f"Format Compliance: {pre_fmt:.1f}% ({pre_grpo_format}/100)")
print(f"Math Accuracy:     {pre_acc:.1f}% ({pre_grpo_correct}/100)\n")

# Free
del base_model_eval
torch.cuda.empty_cache()


Unsloth: DGX Spark detected - `fast_inference=True` is currently broken as of January 2026.
Defaulting to native Unsloth inference.
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.14.1. vLLM: 0.25.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== Running Pre-GRPO Benchmark on 100 GSM8K Test Problems ===
Evaluated 20/100 problems | Current Accuracy: 35.0%
Evaluated 40/100 problems | Current Accuracy: 35.0%
Evaluated 60/100 problems | Current Accuracy: 35.0%
Evaluated 80/100 problems | Current Accuracy: 36.2%
Evaluated 100/100 problems | Current Accuracy: 37.0%

>>> PRE-GRPO BENCHMARK (100 Problems) <<<
Format Compliance: 0.0% (0/100)
Math Accuracy:     37.0% (37/100)



## 4. Run GRPOTrainer using vLLM
We load the tokenizer and define the LoRA parameters. We use `vLLM` inside the trainer with a configured device and VRAM footprint to scale policy rollouts rapidly.

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_REPO_ID,
    max_seq_length=max_seq_length,
    load_in_4bit=False,       # set True for 4-bit QLoRA to save more VRAM
    fast_inference=True,      # enables Unsloth's vLLM-backed rollouts for GRPO
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,  # lower if you hit OOM
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    lora_alpha=64,
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized gradient checkpointing
    random_state=3407,
)

training_args = GRPOConfig(
    use_vllm=True,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=25,
    beta=0.02,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    gradient_accumulation_steps=8,
    per_device_train_batch_size=2,
    num_generations=16,
    temperature=1.0,
    max_completion_length=2048,
    max_steps=850,
    logging_steps=50,
    save_steps=50,
    max_grad_norm=1.0,
    report_to="none",
    output_dir=MODEL_NAME + "-grpo-output",
)

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}


trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward,
        math_step_reward,
        partial_correctness_reward,
        correctness_reward, 
        ],
    args=training_args,
    train_dataset=gsm8k_train,
)

trainer.train()


Unsloth: DGX Spark detected - `fast_inference=True` is currently broken as of January 2026.
Defaulting to native Unsloth inference.
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.14.1. vLLM: 0.25.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


INFO 07-27 10:52:02 [api_utils.py:273] non-default args: {'distributed_executor_backend': 'external_launcher', 'gpu_memory_utilization': 0.3, 'max_num_batched_tokens': 4096, 'max_num_seqs': 16, 'logprobs_mode': 'processed_logprobs', 'disable_log_stats': True, 'model_impl': 'vllm', 'model': 'unsloth/Qwen2.5-0.5B-Instruct'}
INFO 07-27 10:52:04 [model.py:619] Resolved architecture: Qwen2ForCausalLM
INFO 07-27 10:52:04 [model.py:1776] Using max model len 32768
INFO 07-27 10:52:04 [parallel.py:800] Using external launcher for distributed inference.
INFO 07-27 10:52:04 [parallel.py:869] Disabling V1 multiprocessing for external launcher.
INFO 07-27 10:52:04 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 07-27 10:52:04 [vllm.py:1042] Asynchronous scheduling is enabled.
INFO 07-27 10:52:04 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 07-27 10:52:07 [core.py

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

INFO 07-27 10:52:27 [model_runner.py:722] Graph capturing finished in 1 secs, took 0.17 GiB


INFO 07-27 10:52:33 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
INFO 07-27 10:52:34 [core.py:337] init engine (profile, create kv cache, warmup model) took 17.97 s (compilation: 7.59 s)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 850
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 35,192,832 of 529,225,600 (6.65% trained)


WARNING 07-27 10:52:34 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _topk_log_softmax_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
Unsloth: Will smartly offload gradients to save VRAM!
WARNING 07-27 10:52:36 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _rope_embedding_QK. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 07-27 10:52:36 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _rms_layernorm_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 07-27 10:52:36 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _rope_embedding. This causes a latency spike; consider extending warmup to cover this shape/config.


RuntimeError: self and mat2 must have the same dtype, but got Half and Float

In [ ]:
FastLanguageModel.for_inference(model)  # enable fast inference mode

model.save_pretrained_merged(MODEL_NAME + "-grpo-adapter", tokenizer, save_method="merged_16bit")
merged_model = model
print("GRPO training completed and model saved!")


In [ ]:
def plot_training_metrics(
    log_history, 
    window=25, 
    save_path="training_curves_ma.png", 
    figsize=(12, 8), 
    dpi=150
):
    """Extracts and plots training logs with a moving average"""

    logs = pd.DataFrame(log_history)        
    logs = logs[logs["step"].notna()].copy()
    fig, axes = plt.subplots(2, 2, figsize=figsize)

    # 1. Training Loss
    if "loss" in logs:
        logs["loss_ma"] = logs["loss"].rolling(window=window, min_periods=1).mean()
        axes[0, 0].plot(logs["step"], logs["loss"], alpha=0.3, label="Raw Loss")
        axes[0, 0].plot(logs["step"], logs["loss_ma"], color="blue", label=f"MA ({window})")
        axes[0, 0].set_title("Training Loss")
        axes[0, 0].legend()

    # 2. Reward Components
    if "reward" in logs:
        logs["reward_ma"] = logs["reward"].rolling(window=window, min_periods=1).mean()
        axes[0, 1].plot(logs["step"], logs["reward"], alpha=0.3, label="Total Reward (Raw)")
        axes[0, 1].plot(logs["step"], logs["reward_ma"], color="green", label=f"Total Reward MA ({window})")
        
        if "rewards/correctness_reward" in logs:
            logs["correctness_ma"] = logs["rewards/correctness_reward"].rolling(window=window, min_periods=1).mean()
            axes[0, 1].plot(logs["step"], logs["correctness_ma"], label="Correctness MA", alpha=0.8)
            
        if "rewards/format_reward" in logs:
            logs["format_ma"] = logs["rewards/format_reward"].rolling(window=window, min_periods=1).mean()
            axes[0, 1].plot(logs["step"], logs["format_ma"], label="Format MA", alpha=0.8)
            
        axes[0, 1].set_title("Reward Components")
        axes[0, 1].legend()

    # 3. KL Divergence
    if "kl" in logs:
        logs["kl_ma"] = logs["kl"].rolling(window=window, min_periods=1).mean()
        axes[1, 0].plot(logs["step"], logs["kl"], alpha=0.3, label="Raw KL")
        axes[1, 0].plot(logs["step"], logs["kl_ma"], color="orange", label=f"MA ({window})")
        axes[1, 0].set_title("KL Divergence (Policy vs. Reference)")
        axes[1, 0].legend()

    # 4. Gradient Norm
    if "grad_norm" in logs:
        logs["grad_norm_ma"] = logs["grad_norm"].rolling(window=window, min_periods=1).mean()
        axes[1, 1].plot(logs["step"], logs["grad_norm"], alpha=0.3, label="Raw Grad Norm")
        axes[1, 1].plot(logs["step"], logs["grad_norm_ma"], color="red", label=f"MA ({window})")
        axes[1, 1].set_title("Gradient Norm")
        axes[1, 1].legend()

    # Formatting
    for ax in axes.flat:
        ax.set_xlabel("Step")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=dpi)
        
    plt.show()
    
    return fig, axes

plot_training_metrics(trainer.state.log_history)

## 5. Inference Verification
We query our trained model to verify that it generates reasoning and mathematical solutions structured under correct tags.

In [ ]:
merged_model.eval()
merged_model.generation_config.max_length = None

post_grpo_responses_100 = []
post_grpo_correct = 0
post_grpo_format = 0

print("=== Running Post-GRPO Benchmark on 100 GSM8K Test Problems ===")

for i, example in enumerate(eval_dataset_100):
    question = example["question"]
    ground_truth = extract_hash_answer(example["answer"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs["input_ids"]
    input_ids = input_ids.to("cuda")

    with torch.no_grad():
        outputs = merged_model.generate(
            input_ids=input_ids, max_new_tokens=2048, pad_token_id=tokenizer.eos_token_id
        )

    post_resp = tokenizer.decode(
        outputs[0][input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    post_grpo_responses_100.append(post_resp)
    ext_ans = extract_last_number(post_resp)

    if ext_ans == ground_truth:
        post_grpo_correct += 1

    if re.match(pattern, post_resp):
        post_grpo_format += 1

    if (i + 1) % 20 == 0:
        print(
            f"Evaluated {i + 1}/100 problems | Current Accuracy: {post_grpo_correct / (i + 1) * 100:.1f}%"
        )

post_acc = (post_grpo_correct / 100) * 100
post_fmt = (post_grpo_format / 100) * 100
acc_delta = post_acc - pre_acc
fmt_delta = post_fmt - pre_fmt

print("\n" + "=" * 60)
print(">>> FINAL GSM8K 100-PROBLEM BENCHMARK RESULTS <<<")
print(f"Pre-GRPO Format Compliance:  {pre_fmt:.1f}%")
print(f"Post-GRPO Format Compliance: {post_fmt:.1f}%")
print(f"Format Compliance Delta:    {'+' if fmt_delta >= 0 else ''}{fmt_delta:.1f}%\n")
print(f"Pre-GRPO Math Accuracy:      {pre_acc:.1f}%")
print(f"Post-GRPO Math Accuracy:     {post_acc:.1f}%")
print(f"Math Accuracy Delta:        {'+' if acc_delta >= 0 else ''}{acc_delta:.1f}%")
print("=" * 60 + "\n")
print("=== Detailed 5-Example Side-by-Side Comparison ===")

for i in range(5):
    example = eval_dataset_100[i]
    question = example["question"]
    ground_truth = extract_hash_answer(example["answer"])
    print(f"=== Test Example {i + 1} ===")
    print(f"Question: {question}")
    print(f"Ground Truth Answer: {ground_truth}")
    print(
        f"\033[91mBefore GRPO (Pre-trained Model):\033[0m\n{pre_grpo_responses_100[i]}\n"
    )
    print(
        f"\033[92mAfter GRPO (RL Aligned Model):\033[0m\n{post_grpo_responses_100[i]}\n"
    )
    print("-" * 80 + "\n")